In [1]:
# Imports

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.dataset as ds

import calendar


In [2]:
# Load

dataset = ds.dataset("data/weather.parquet", format = "parquet")

table = dataset.to_table(columns = ["time", "precipitation"])

df = table.to_pandas()
df["time"] = df["time"].dt.tz_convert("America/Edmonton")

df = df.loc[df["time"].between("2025-01-01", "2026-01-01", inclusive = "left")]

df.head()


In [3]:
# Inspect

# Structure
print(df["time"].min(), "to", df["time"].max(), "\n")
print("Shape:", df.shape, "\n")
print("Dtypes:")
print(df.dtypes, "\n")

# Missingness
print("Any Nulls:", df.isna().any().any(), "\n")

# Duplicates
print("Duplicate Rows:", df.duplicated().any())
print("Duplicate Times:", df["time"].duplicated().any(), "\n")

# Temporal Sanity
print("Non-Monotonic Times:", not df["time"].is_monotonic_increasing)
print("Unexpected Time Deltas:", df["time"].diff().dropna().ne(pd.Timedelta(hours = 1)).any(), "\n")

# Value Sanity
display(df[["precipitation"]].describe().T)


2025-01-01 00:00:00-07:00 to 2025-12-31 23:00:00-07:00
Shape: (8938, 2)
Dtypes:
time             datetime64[ns, America/Edmonton]
precipitation                             float64
dtype: object
Any Nulls: False
Duplicate Rows: False
Duplicate Times: True
Non-Monotonic Times: False
Unexpected Time Deltas: True


In [4]:
# Additional Inspection - Duplicate Times, no Duplicate Rows, and Unexpected Time Deltas

Expected = 365 * 24
Actual = len(df)

print("Conflicting Rows:", 2 *(Actual - Expected))
print("Conflicting Percentage:", round((100* 2 * (Actual - Expected) / Expected), 3), "%")
print("Most Duplicate Times:", df["time"].value_counts().max(), "\n")

display(df.loc[df["time"].duplicated(keep = False) == True].head())


Conflicting Rows: 356
Conflicting Percentage: 4.064 %
Most Duplicate Times: 2


In [5]:
# Additional Inspection - Temporal Distribution of Duplicates

display(df.loc[df["time"].eq("2025-01-03 00:00:00-07:00")])

display(df[df["time"].duplicated(keep = False)].groupby(df["time"].dt.month).size())

display(df[df["time"].duplicated(keep=False)].groupby(df["time"].dt.date).size().sort_values(ascending=False).head(12))


time
1      10
2       2
3      88
4      10
5      14
6     174
7       6
8      10
9      10
10      8
11     14
12     10
dtype: int64


time
2025-03-08    48
2025-06-01    48
2025-06-11    48
2025-06-02    34
2025-03-07    34
2025-06-10    20
2025-06-12    14
2025-10-25     4
2025-06-14     4
2025-08-12     4
2025-09-28     2
2025-08-24     2
dtype: int64


In [6]:
print("Percentage Duplicated June Times:", round((174/2) / (30 * 24), 4) * 100, "%")
print("Percentage Duplicated March Times:", round((88/2) / (31 * 24), 4) * 100, "%")
print("Percentage Duplicated May Times:", round((14/2) / (31 * 24), 5) * 100, "%")


Percentage Duplicated June Times: 12.08 %
Percentage Duplicated March Times: 5.91 %
Percentage Duplicated May Times: 0.941 %


In [7]:
# Clean - Drop Duplicates, Replace times with monthly average precipitation, Accept Interpretability Issues

clean = df.loc[~df["time"].duplicated(keep=False)].copy()

clean = clean.set_index("time").sort_index().asfreq("h").reset_index()

monthly_avg = clean.groupby(clean["time"].dt.month)["precipitation"].transform("mean")

clean["precipitation"] = clean["precipitation"].fillna(monthly_avg)

display(clean[["precipitation"]].describe().T)


In [8]:
# Transform

output_df = (clean.assign(month = lambda x: x["time"].dt.month)
                  .groupby(["month"], as_index = False)
                  .agg(avg_prec = ("precipitation", "mean"),
                       total_prec = ("precipitation", "sum"),
                       max_prec = ("precipitation", "max"))
            )

output_df


In [9]:
# Transform - Annual Concentration

concentration_df = (
    output_df[["month", "total_prec"]]
    .assign(
        share_pct=lambda x: (
            x["total_prec"] / x["total_prec"].sum() * 100
        )
    )
)

annual_total = concentration_df["total_prec"].sum()

top3_df = concentration_df.nlargest(3, "total_prec")
top3_share = top3_df["share_pct"].sum()

# Validate

assert len(concentration_df) == 12
assert concentration_df["month"].nunique() == 12
assert concentration_df["total_prec"].ge(0).all()
assert abs(concentration_df["share_pct"].sum() - 100) < 1e-9

print(f"Estimated annual total: {annual_total:.1f} mm")
print(f"Top three months: {top3_share:.1f}%")

concentration_df


Estimated annual total: 529.8 mm
Top three months: 59.5%


In [10]:
# Visualize

# Figure

plt.rcParams["font.family"] = "Avenir Next"
plt.rcParams["font.weight"] = 500

fig, ax = plt.subplots(figsize = (12, 6), dpi = 300)

fig.subplots_adjust(
    left = 0.13,
    right = 0.96,
    top = 0.80,
    bottom = 0.15
)

STRUCTURE = "#777777"
HIGHLIGHT = "#376F91"
MUTED = "#D0D4D8"
BACKGROUND = "#FFFFFF"

fig.patch.set_facecolor(BACKGROUND)

# Plot

colors = [HIGHLIGHT if month in [5, 6, 7] else MUTED for month in concentration_df["month"]]

bars = ax.bar(concentration_df["month"], concentration_df["share_pct"], color = colors, width = 0.72)

for month, bar in zip(concentration_df["month"], bars):
    if month == 6:
        bar.set_hatch("///")
        bar.set_edgecolor(STRUCTURE)

# Scale & Axes

ax.set_xlim(0.4, 12.6)
ax.set_ylim(0, 30)

ax.set_ylabel("SHARE OF ESTIMATED ANNUAL TOTAL (%)", fontsize = 10, fontweight = 500,
              color = STRUCTURE, labelpad = 12, loc = "top")

# Ticks

ax.set_xticks(range(1, 13))
ax.set_xticklabels(calendar.month_abbr[1:13])
ax.tick_params(axis = "x", labelsize = 10, length = 0, labelcolor = STRUCTURE)

ax.tick_params(axis = "y", labelsize = 10, length = 0, labelcolor = STRUCTURE)

# Styling

ax.bar_label(bars, labels = [f"{share:.1f}%" for share in concentration_df["share_pct"]],
             padding = 4, fontsize = 9, color = STRUCTURE)

ax.yaxis.grid(True, color = STRUCTURE, alpha = 0.18, linewidth = 0.7)
ax.set_axisbelow(True)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(STRUCTURE)

# Composition

fig.suptitle("Calgary's Monthly Precipitation Concentration — 2025",
             x = 0.08, y = 0.96, ha = "left", fontsize = 17, fontweight = 600)

fig.text(0.0, 0.875, f"May–July contributed {top3_share:.1f}% of estimated annual precipitation",
         fontsize = 11, color = HIGHLIGHT)

fig.text(0.08, 0.075, "Monthly totals include mean-imputed conflicting hours; June has 87 affected hours (hatched).",
         fontsize = 9, color = STRUCTURE)

# Output

fig.savefig("outputs/precipitation_concentration_2025.svg", bbox_inches = "tight", pad_inches = 0.1)
plt.show()
